# 04 — Premiums & greeks exploration (Week 3, step 1)

**Objective.** Gather the facts to settle the S3 fork (README §8bis): build the straddles with
the **surface premiums** (`impl_premium` from vsurfd) or **recompute Black–Scholes** from the IV
(+ `zerocd` rates, dividends)?

**Questions:**
1. What exactly does `vsurfd` contain (available columns: premium? strike? dispersion?)
2. What do `zerocd` (zero-coupon rate) and the spot price table (`secprd`) look like?
3. **Decisive test**: does BS(iv, impl_strike, spot, r, T) reproduce `impl_premium`? With what implied
   dividend yield?

Test date: **2024-06-28** (last rebalancing of June 2024, master-calendar day).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from dispersion.data.wrds_client import get_connection

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
db = get_connection()
print("connection OK")

Loading library list...


Done
connection OK


In [2]:
# Candidate tables in optionm: spot price, rates, forwards
tabs = db.raw_sql("""
SELECT table_name FROM information_schema.tables
WHERE table_schema = 'optionm'
  AND (table_name LIKE 'secprd%%' OR table_name LIKE 'zerocd%%' OR table_name LIKE 'fwdprd%%')
ORDER BY table_name
""")
print(tabs.to_string())

    table_name
0   fwdprd1996
1   fwdprd1997
2   fwdprd1998
3   fwdprd1999
4   fwdprd2000
5   fwdprd2001
6   fwdprd2002
7   fwdprd2003
8   fwdprd2004
9   fwdprd2005
10  fwdprd2006
11  fwdprd2007
12  fwdprd2008
13  fwdprd2009
14  fwdprd2010
15  fwdprd2011
16  fwdprd2012
17  fwdprd2013
18  fwdprd2014
19  fwdprd2015
20  fwdprd2016
21  fwdprd2017
22  fwdprd2018
23  fwdprd2019
24  fwdprd2020
25  fwdprd2021
26  fwdprd2022
27  fwdprd2023
28  fwdprd2024
29  fwdprd2025
30      secprd
31  secprd1996
32  secprd1997
33  secprd1998
34  secprd1999
35  secprd2000
36  secprd2001
37  secprd2002
38  secprd2003
39  secprd2004
40  secprd2005
41  secprd2006
42  secprd2007
43  secprd2008
44  secprd2009
45  secprd2010
46  secprd2011
47  secprd2012
48  secprd2013
49  secprd2014
50  secprd2015
51  secprd2016
52  secprd2017
53  secprd2018
54  secprd2019
55  secprd2020
56  secprd2021
57  secprd2022
58  secprd2023
59  secprd2024
60  secprd2025
61      zerocd


In [3]:
# Full schema of the vsurfd surface (year 2024)
cols = db.raw_sql("""
SELECT column_name, data_type FROM information_schema.columns
WHERE table_schema = 'optionm' AND table_name = 'vsurfd2024'
ORDER BY ordinal_position
""")
print(cols.to_string())

       column_name          data_type
0            secid   double precision
1             date               date
2             days   double precision
3            delta   double precision
4  impl_volatility   double precision
5      impl_strike   double precision
6     impl_premium   double precision
7       dispersion   double precision
8          cp_flag  character varying


In [4]:
# SPX sample: 91 days, ±50 delta, 2024-06-28 — all columns
spx = db.raw_sql("""
SELECT * FROM optionm.vsurfd2024
WHERE secid = 108105 AND days = 91 AND delta IN (50, -50) AND date = '2024-06-28'
""")
spx

,secid,date,days,delta,impl_volatility,impl_strike,impl_premium,dispersion,cp_flag
0,108105.0,2024-06-28,91.0,-50.0,0.113183,5531.409,127.7991,0.006892,P
1,108105.0,2024-06-28,91.0,50.0,0.121323,5530.623,127.2951,0.007312,C


In [5]:
# Same for the largest market cap of the universe at this rebalancing
w = pd.read_parquet(ROOT / "data" / "processed" / "weights.parquet")
top = w[w["rebalance_date"] == "2024-06-28"].nsmallest(1, "rnk")
sec_top = int(top["secid"].iloc[0])
print(top[["permno", "secid", "weight", "rnk"]].to_string(index=False))
comp = db.raw_sql(f"""
SELECT * FROM optionm.vsurfd2024
WHERE secid = {sec_top} AND days = 91 AND delta IN (50, -50) AND date = '2024-06-28'
""")
comp

 permno  secid   weight  rnk
  10107 107525  0.09541    1


,secid,date,days,delta,impl_volatility,impl_strike,impl_premium,dispersion,cp_flag
0,107525.0,2024-06-28,91.0,-50.0,0.211219,452.5007,19.46269,0.006059,P
1,107525.0,2024-06-28,91.0,50.0,0.228016,455.1952,18.95197,0.007014,C


In [6]:
# OptionMetrics zero-coupon curve around 91 days
zc = db.raw_sql("SELECT * FROM optionm.zerocd WHERE date = '2024-06-28' ORDER BY days LIMIT 12")
zc

,date,days,rate
0,2024-06-28,10.0,5.397591
1,2024-06-28,30.0,5.426773
2,2024-06-28,60.0,5.465693
3,2024-06-28,91.0,5.499974
4,2024-06-28,122.0,5.528427
5,2024-06-28,152.0,5.55061
6,2024-06-28,182.0,5.567719
7,2024-06-28,273.0,5.590523
8,2024-06-28,365.0,5.57328
9,2024-06-28,547.0,5.441071


In [7]:
# SPX spot price (secprd table, annual or single depending on the subscription)
cands = [t for t in tabs["table_name"] if t.startswith("secprd")]
name = "secprd2024" if "secprd2024" in cands else (cands[0] if cands else None)
print("selected price table:", name)
spot = db.raw_sql(f"SELECT * FROM optionm.{name} WHERE secid = 108105 AND date = '2024-06-28'")
spot

selected price table: secprd2024


,secid,date,low,high,close,volume,return,cfadj,open,cfret,shrout
0,108105.0,2024-06-28,5451.12,5523.64,5460.48,0.0,-0.004084,1.0,5488.48,1.0,0.0


In [8]:
# DECISIVE TEST: does Black-Scholes reproduce impl_premium?
from scipy.stats import norm
from scipy.optimize import brentq

S = abs(float(spot["close"].iloc[0]))
T = 91 / 365.0
r = float(np.interp(91, zc["days"], zc["rate"])) / 100   # % -> decimal, continuously compounded
print(f"S = {S:.2f} | r(91d) = {r:.4%} | T = {T:.4f}")

def bs(S, K, sig, T, r, q, cp):
    d1 = (np.log(S / K) + (r - q + 0.5 * sig**2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    if cp == "C":
        return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)

for label, df in (("SPX", spx), ("top component", comp)):
    if label == "top component":
        s_row = db.raw_sql(f"SELECT * FROM optionm.{name} WHERE secid = {sec_top} AND date = '2024-06-28'")
        S_loc = abs(float(s_row["close"].iloc[0]))
    else:
        S_loc = S
    print(f"\n--- {label} (spot {S_loc:.2f}) ---")
    for _, row in df.iterrows():
        K, sig_iv, prem = float(row["impl_strike"]), float(row["impl_volatility"]), float(row["impl_premium"])
        cp = str(row["cp_flag"]).strip()
        p0 = bs(S_loc, K, sig_iv, T, r, 0.0, cp)
        try:
            q_star = brentq(lambda q: bs(S_loc, K, sig_iv, T, r, q, cp) - prem, -0.10, 0.15)
        except ValueError:
            q_star = np.nan
        print(f"{cp}: impl_premium = {prem:9.3f} | BS(q=0) = {p0:9.3f} ({100*(p0/prem-1):+5.1f}%) | implied q = {q_star:+.2%}")

S = 5460.48 | r(91d) = 5.5000% | T = 0.2493

--- SPX (spot 5460.48) ---
P: impl_premium =   127.799 | BS(q=0) =   120.857 ( -5.4%) | implied q = +1.04%
C: impl_premium =   127.295 | BS(q=0) =   134.487 ( +5.6%) | implied q = +1.04%

--- top component (spot 446.95) ---
P: impl_premium =    19.463 | BS(q=0) =    18.479 ( -5.1%) | implied q = +1.83%
C: impl_premium =    18.952 | BS(q=0) =    19.329 ( +2.0%) | implied q = +0.67%


In [9]:
# HISTORICAL coverage of impl_premium on OUR fixed grid (91d, ±50δ) — we commit over 29 years
years = [1996, 2005, 2015]
rows = []
for y in years:
    # SPX, full year
    a = db.raw_sql(f"""
        SELECT COUNT(*) AS n, COUNT(impl_premium) AS n_prem, COUNT(impl_strike) AS n_strike
        FROM optionm.vsurfd{y}
        WHERE secid = 108105 AND days = 91 AND delta IN (50, -50)
    """)
    # point-in-time top-100 universe at the mid-year rebalancing, on the rebalancing day
    reb = sorted(d for d in w["rebalance_date"].unique() if pd.Timestamp(d).year == y)[1]
    secs = w.loc[w["rebalance_date"] == reb, "secid"].dropna().astype(int).tolist()
    b = db.raw_sql(f"""
        SELECT COUNT(*) AS n, COUNT(impl_premium) AS n_prem
        FROM optionm.vsurfd{y}
        WHERE secid IN ({', '.join(map(str, secs))}) AND days = 91 AND delta IN (50, -50)
          AND date = '{pd.Timestamp(reb).date()}'
    """)
    rows.append((y, int(a["n"][0]), int(a["n_prem"][0]), int(a["n_strike"][0]),
                 str(pd.Timestamp(reb).date()), len(secs), int(b["n"][0]), int(b["n_prem"][0])))
cov = pd.DataFrame(rows, columns=["year", "SPX rows", "SPX prem", "SPX strike",
                                  "rebal", "n secids", "universe rows", "universe prem"])
print("expected universe rows = 2 × n secids (call + put)")
cov

expected universe rows = 2 × n secids (call + put)


,year,SPX rows,SPX prem,SPX strike,rebal,n secids,universe rows,universe prem
0,1996,504,504,504,1996-06-28,100,198,198
1,2005,504,504,504,2005-06-30,100,200,200
2,2015,504,504,504,2015-06-30,100,200,200


## Findings & framing by the already-settled choices

**Locked choices, NOT reopened here**: extraction grid = **ATM ±50δ, 91 d** (locked in S1 — it is
exactly the grid sampled above); ρ_implied in **ATM, MFIV discarded** (S2 fork, README
§5); **unconditional v0 first** and **relative greeks + unit test** (DMV conventions, README
§8bis); the **costs** fork (opprcd vs parametric) is distinct and stays open — nothing here
preempts it.

**Facts established by this notebook:**
1. `vsurfd` carries `impl_premium` + `impl_strike` (+ `dispersion`) on exactly our fixed grid →
   the premiums exist without changing a single design parameter.
2. **Historical coverage** (cell above): premiums present from 1996, on SPX as on
   the point-in-time universe.
3. **BS(q=0) misses the premiums by ±5%** (underprices puts, overprices calls); the implied q for
   SPX = **1.04% identical on both legs** proves that `impl_premium` is forward-consistent.
   Recomputing them oneself would require the dividend yield (index) and American exercise / discrete
   dividends (components) — heavy and fragile.
4. `zerocd` (rates), `fwdprd` (forwards 1996→2025) and `secprd` (spots) are available → the analytic
   greeks are cleanly computable (IV + strike + forward + rate).

**→ Proposed fork (to validate): premiums = `impl_premium` (re-extraction, 2 extra columns in
`get_iv`); greeks = BS/Black-76 from IV + forward (`fwdprd` or call-put parity) + `zerocd`.**
The marking of ageing positions (91d → 28d: surface interpolation vs DMV-style holding-period) =
a `DispersionEngine` design question, raised at the next step.